# SAGE-Price Model

**Score-weighted Asset-pricing Guided Ensemble**  
이 노트북은 5개 전통 자산가격결정모형과 가치평가모형을 백테스트한 뒤, 과거 기대주가 예측 성과를 비중치로 변환하여 1년 뒤 최종 기대 주가를 계산합니다.

핵심 산식은 `최종 기대 주가 = Σ(모형별 기대 주가 × 모형별 성과 기반 비중치)`입니다.

셀은 위에서 아래로 순서대로 실행하면 됩니다.


## 1. 기본 설정

- 한국 주식은 Yahoo Finance 기준으로 삼성전자 `005930.KS`처럼 씁니다.
- 미국 주식은 `AAPL`, `MSFT`, `NVDA`처럼 씁니다.
- `USE_SYNTHETIC = True`로 바꾸면 인터넷 없이 샘플 데이터로 실행합니다.

In [ ]:
from datetime import date, timedelta
from pathlib import Path
from types import SimpleNamespace
import sys

from IPython.display import Markdown, display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "stock_range_model").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stock_range_model.cli import build_model_context
from stock_range_model.data import download_yahoo_prices, generate_synthetic_prices
from stock_range_model.ensemble import forecast_with_weighted_models
from stock_range_model.models import default_models

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

TICKER = "005930.KS"   # 예: 삼성전자. 미국 주식은 "AAPL"처럼 입력
YEARS = 10
CONFIDENCE = 0.80
USE_SYNTHETIC = False

def format_price(value):
    return f"{value:,.2f}"

def format_pct(value):
    return f"{value:.2%}"


## 2. 가격 데이터 불러오기

In [11]:
if USE_SYNTHETIC:
    prices = generate_synthetic_prices(years=YEARS, seed=42)
else:
    end = date.today()
    start = end - timedelta(days=int(YEARS * 365.25))
    prices = download_yahoo_prices(TICKER, start=start, end=end)

prices = prices.dropna()

print(f"데이터 개수: {len(prices):,}")
print(f"시작일: {prices.index.min().date()}")
print(f"종료일: {prices.index.max().date()}")
print(f"최근 가격: {prices.iloc[-1]:,.2f}")

prices.tail()

데이터 개수: 2,447
시작일: 2016-06-24
종료일: 2026-06-24
최근 가격: 325,000.00


2026-06-18   362,500.0000
2026-06-19   354,000.0000
2026-06-22   353,500.0000
2026-06-23   310,000.0000
2026-06-24   325,000.0000
Name: 005930.KS, dtype: float64

## 3. 가격 그래프

가격 흐름을 먼저 확인한 뒤, 이어서 SAGE-Price Model의 1년 뒤 기대주가를 계산합니다.


In [ ]:
import matplotlib.pyplot as plt

ax = prices.plot(figsize=(12, 5), title=f"{TICKER} price history")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted close")
plt.show()


## 4. SAGE-Price Model 실행 및 최종 요약

아래 셀은 5개 모형을 백테스트하고, 과거 성과에 따라 비중치를 계산한 뒤 최종 기대주가를 산출합니다.


In [ ]:
context_args = SimpleNamespace(
    ticker=None if USE_SYNTHETIC else TICKER,
    market_ticker=None,
    risk_free_rate=None,
    required_return=None,
    book_value_per_share=None,
    roe=None,
    terminal_growth=None,
    dividend_yield_fallback=0.02,
    price_to_book_fallback=1.5,
)
model_context = build_model_context(context_args, prices)
models = default_models(context=model_context)

forecast = forecast_with_weighted_models(
    prices,
    models=models,
    min_train_days=252 * 3,
    horizon_days=252,
    step_days=252,
    confidence=CONFIDENCE,
)

direction = "상승" if forecast.expected_return >= 0 else "하락"
display(Markdown(
    f"### 최종 결과: 1년 뒤 기대주가 {format_price(forecast.expected)}"
    f"\n\n현재 주가 {format_price(forecast.current_price)} 대비 "
    f"**{format_pct(forecast.expected_return)} {direction}**을 예상합니다."
))

summary_view = pd.DataFrame([
    {"항목": "현재 주가", "값": format_price(forecast.current_price)},
    {"항목": "SAGE 최종 기대주가", "값": format_price(forecast.expected)},
    {"항목": "예상 수익률", "값": format_pct(forecast.expected_return)},
    {"항목": "참고 하단 가격", "값": format_price(forecast.lower)},
    {"항목": "참고 상단 가격", "값": format_price(forecast.upper)},
    {"항목": "신뢰수준", "값": format_pct(CONFIDENCE)},
    {"항목": "예측 기간", "값": "1년, 약 252거래일"},
])
display(summary_view)


## 5. 모형별 성과 기반 비중치

`weight`는 최종 기대주가 계산에 들어간 모형별 영향력입니다. 값이 클수록 과거 백테스트에서 상대적으로 더 좋은 성과를 보인 모형입니다.


In [ ]:
weights_view = forecast.weights[[
    "model",
    "tests",
    "avg_absolute_relative_expected_error",
    "expected_rmse_relative",
    "coverage_rate",
    "weight",
]].rename(columns={
    "model": "모형",
    "tests": "검증 횟수",
    "avg_absolute_relative_expected_error": "평균 절대 오차율",
    "expected_rmse_relative": "RMSE 오차율",
    "coverage_rate": "범위 적중률",
    "weight": "최종 비중",
})

display(
    weights_view.style.format({
        "평균 절대 오차율": "{:.2%}",
        "RMSE 오차율": "{:.2%}",
        "범위 적중률": "{:.2%}",
        "최종 비중": "{:.2%}",
    }).bar(subset=["최종 비중"], color="#9DC3E6")
)

ax = weights_view.sort_values("최종 비중").plot.barh(
    x="모형",
    y="최종 비중",
    figsize=(10, 4),
    legend=False,
    color="#2E74B5",
    title="Model weights in SAGE-Price Model",
)
ax.set_xlabel("Weight")
ax.set_ylabel("")
plt.show()


## 6. 모형별 1년 뒤 기대주가

각 모형이 제시한 기대주가와, 그 값이 최종 기대주가에 얼마나 기여했는지 확인합니다.


In [ ]:
predictions_view = forecast.model_predictions[[
    "model",
    "expected",
    "expected_return",
    "weight",
    "weighted_expected",
    "weighted_expected_return",
]].rename(columns={
    "model": "모형",
    "expected": "모형 기대주가",
    "expected_return": "모형 예상수익률",
    "weight": "비중",
    "weighted_expected": "최종 기대주가 기여분",
    "weighted_expected_return": "최종 수익률 기여분",
})

display(
    predictions_view.style.format({
        "모형 기대주가": "{:,.2f}",
        "모형 예상수익률": "{:.2%}",
        "비중": "{:.2%}",
        "최종 기대주가 기여분": "{:,.2f}",
        "최종 수익률 기여분": "{:.2%}",
    }).bar(subset=["비중"], color="#D9EAF7")
)


## 7. 최종 기대주가 계산 검산

SAGE-Price Model의 최종 기대주가는 아래 공식으로 계산됩니다.

```text
최종 기대주가 = Σ(모형별 기대주가 × 모형별 비중치)
```


In [ ]:
weighted_average_check = pd.DataFrame([
    {
        "검산 항목": "Σ(모형별 기대주가 × 비중치)",
        "직접 합산값": forecast.model_predictions["weighted_expected"].sum(),
        "최종 기대주가": forecast.expected,
        "차이": forecast.model_predictions["weighted_expected"].sum() - forecast.expected,
    },
])

display(weighted_average_check.style.format({
    "직접 합산값": "{:,.6f}",
    "최종 기대주가": "{:,.6f}",
    "차이": "{:,.10f}",
}))


## 8. 백테스트 결과 확인

10년치 데이터를 모두 사용하지만, 그래프에 10개년 전체가 그대로 표시되는 것은 아닙니다. 최초 약 3년은 첫 예측을 위한 학습 구간으로 사용하고, 각 검증은 1년 뒤 실제 가격이 필요하기 때문입니다.

즉, 마지막 백테스트 표와 그래프는 `전체 데이터 기간`이 아니라 `학습 구간 이후 실제 1년 뒤 가격까지 확인 가능한 검증 시점`을 보여줍니다.


In [ ]:
ensemble_view = forecast.ensemble_backtest.copy()

display(Markdown(
    f"백테스트 비교 횟수: **{len(ensemble_view)}회**  \n"
    f"설정: 전체 데이터 {YEARS}년, 최초 학습기간 3년, 예측기간 1년, 검증 간격 1년"
))

ax = ensemble_view.plot(
    x="target_date",
    y=["actual_price", "ensemble_expected"],
    figsize=(12, 5),
    marker="o",
    title="Backtest: actual price vs SAGE expected price",
)
ax.set_xlabel("Target date")
ax.set_ylabel("Price")
ax.legend(["Actual price", "SAGE expected price"])
plt.show()

backtest_table = ensemble_view[[
    "origin_date",
    "target_date",
    "current_price",
    "ensemble_expected",
    "actual_price",
    "absolute_percentage_error",
]].rename(columns={
    "origin_date": "예측 기준일",
    "target_date": "실제 비교일",
    "current_price": "당시 주가",
    "ensemble_expected": "SAGE 기대주가",
    "actual_price": "실제 주가",
    "absolute_percentage_error": "절대 오차율",
})

display(backtest_table.style.format({
    "당시 주가": "{:,.2f}",
    "SAGE 기대주가": "{:,.2f}",
    "실제 주가": "{:,.2f}",
    "절대 오차율": "{:.2%}",
}))
